# Sel Tahmin Veri Seti ile Regresyon

In [1]:
import os
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

In [2]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
HEDEF = 'FloodProbability'

In [3]:
train

,id,MonsoonIntensity,TopographyDrainage,RiverManagement,Deforestation,Urbanization,ClimateChange,DamsQuality,Siltation,AgriculturalPractices,...,DrainageSystems,CoastalVulnerability,Landslides,Watersheds,DeterioratingInfrastructure,PopulationScore,WetlandLoss,InadequatePlanning,PoliticalFactors,FloodProbability
0,0,5,8,5,8,6,4,4,3,3,...,5,3,3,5,4,7,5,7,3,0.445
1,1,6,7,4,4,8,8,3,5,4,...,7,2,0,3,5,3,3,4,3,0.450
2,2,6,5,6,7,3,7,1,5,4,...,7,3,7,5,6,8,2,3,3,0.530
3,3,3,4,6,5,4,8,4,7,6,...,2,4,7,4,4,6,5,7,5,0.535
4,4,5,3,2,6,4,4,3,3,3,...,2,2,6,6,4,1,2,3,5,0.415
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1117952,1117952,3,3,4,10,4,5,5,7,10,...,7,8,7,2,2,1,4,6,4,0.495
1117953,1117953,2,2,4,3,9,5,8,1,3,...,9,4,4,3,7,4,9,4,5,0.480
1117954,1117954,7,3,9,4,6,5,9,1,3,...,5,5,5,5,6,5,5,2,4,0.485
1117955,1117955,7,3,3,7,5,2,3,4,6,...,6,8,5,3,4,6,7,6,4,0.495


In [4]:
test

,id,MonsoonIntensity,TopographyDrainage,RiverManagement,Deforestation,Urbanization,ClimateChange,DamsQuality,Siltation,AgriculturalPractices,...,IneffectiveDisasterPreparedness,DrainageSystems,CoastalVulnerability,Landslides,Watersheds,DeterioratingInfrastructure,PopulationScore,WetlandLoss,InadequatePlanning,PoliticalFactors
0,1117957,4,6,3,5,6,7,8,7,8,...,8,5,7,5,6,3,6,4,4,5
1,1117958,4,4,2,9,5,5,4,7,5,...,2,4,7,4,5,1,7,4,4,3
2,1117959,1,3,6,5,7,2,4,6,4,...,7,9,2,5,5,2,3,6,8,3
3,1117960,2,4,4,6,4,5,4,3,4,...,7,8,4,6,7,6,4,2,4,4
4,1117961,6,3,2,4,6,4,5,5,3,...,4,3,2,6,4,6,8,4,5,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
745300,1863257,5,4,8,3,5,4,4,5,5,...,5,6,1,3,5,6,4,4,6,6
745301,1863258,4,4,2,12,4,3,4,3,5,...,3,7,4,4,3,5,5,3,5,4
745302,1863259,5,7,9,5,5,6,7,5,5,...,6,11,3,11,4,5,9,5,5,4
745303,1863260,4,7,6,3,5,2,3,8,6,...,6,6,8,6,2,3,8,7,5,5


In [5]:
train.isnull().sum()

id                                 0
MonsoonIntensity                   0
TopographyDrainage                 0
RiverManagement                    0
Deforestation                      0
Urbanization                       0
ClimateChange                      0
DamsQuality                        0
Siltation                          0
AgriculturalPractices              0
Encroachments                      0
IneffectiveDisasterPreparedness    0
DrainageSystems                    0
CoastalVulnerability               0
Landslides                         0
Watersheds                         0
DeterioratingInfrastructure        0
PopulationScore                    0
WetlandLoss                        0
InadequatePlanning                 0
PoliticalFactors                   0
FloodProbability                   0
dtype: int64

In [6]:
test.isnull().sum()

id                                 0
MonsoonIntensity                   0
TopographyDrainage                 0
RiverManagement                    0
Deforestation                      0
Urbanization                       0
ClimateChange                      0
DamsQuality                        0
Siltation                          0
AgriculturalPractices              0
Encroachments                      0
IneffectiveDisasterPreparedness    0
DrainageSystems                    0
CoastalVulnerability               0
Landslides                         0
Watersheds                         0
DeterioratingInfrastructure        0
PopulationScore                    0
WetlandLoss                        0
InadequatePlanning                 0
PoliticalFactors                   0
dtype: int64

In [7]:
#  feature engineer (Row-wise Statistics)
#Her bir satır için Toplam Puan (f_sum) ve Ortalama Puan (f_mean) sütunları üretmek.
#Her bir bölgenin faktör çeşitliliğini ölçmek için satır bazlı Standart Sapma (f_std) ve Medyan (f_median) değerlerini hesaplamak.
def kopya_ekle(df):
    df_copy = df.copy()
    # Tahminde kullanılacak ham sütunları seçelim (id ve hedef hariç)
    ham_sutunlar = [col for col in df_copy.columns if col not in ['id', HEDEF]]
    
    # Satır bazlı istatistiksel özetleri yeni sütun olarak ekliyoruz
    df_copy['f_sum'] = df_copy[ham_sutunlar].sum(axis=1)
    df_copy['f_mean'] = df_copy[ham_sutunlar].mean(axis=1)
    df_copy['f_std'] = df_copy[ham_sutunlar].std(axis=1)
    df_copy['f_median'] = df_copy[ham_sutunlar].median(axis=1)
    df_copy['f_max'] = df_copy[ham_sutunlar].max(axis=1)
    df_copy['f_min'] = df_copy[ham_sutunlar].min(axis=1)
    return df_copy
    

In [8]:
train_fe = kopya_ekle(train)
test_fe = kopya_ekle(test)

In [9]:
# Girdileri (X) ve Hedefi (y) Kesin Olarak Ayırma
giris_sutunlari = [kolon for kolon in train_fe.columns if kolon not in ['id', HEDEF]]

X = train_fe[giris_sutunlari]
y = train_fe[HEDEF]
X_test = test_fe[giris_sutunlari]


In [10]:
# Yapay sinir ağlarının kararlı çalışması için verileri standart ölçeğe getirme
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_test_scaled = scaler.transform(X_test)


In [11]:
# Keras Derin Öğrenme Modeli Tasarımı
# Regresyon görevinde R2 skorunu zirveye taşımak için optimize edilmiş mimari
model = models.Sequential([
    layers.Input(shape=(X_scaled.shape[1],)),
    layers.Dense(256, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.2),
    layers.Dense(128, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.2),
    layers.Dense(64, activation='relu'),
    layers.Dense(1) # Olasılık tahmini (Regresyon çıktısı) için tek nöron
])
# R2 skoru doğrudan MSE kaybı azaltılarak optimize edilir
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss='mse')


In [12]:
# Aşırı öğrenmeyi engellemek için erken durdurma (Early Stopping)
early_stopping = callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

In [13]:
# Modeli eğitme
history = model.fit(X_scaled, y,epochs=20,batch_size=1024) # Büyük veri setlerinde hızlı eğitim için yüksek batch sizevalidation_split=0.1,callbacks=[early_stopping],verbose=1)


Epoch 1/20
1092/1092 ━━━━━━━━━━━━━━━━━━━━ 24s 18ms/step - loss: 0.0530
Epoch 2/20
1092/1092 ━━━━━━━━━━━━━━━━━━━━ 20s 19ms/step - loss: 0.0021
Epoch 3/20
1092/1092 ━━━━━━━━━━━━━━━━━━━━ 19s 18ms/step - loss: 0.0013
Epoch 4/20
1092/1092 ━━━━━━━━━━━━━━━━━━━━ 19s 17ms/step - loss: 9.7183e-04
Epoch 5/20
1092/1092 ━━━━━━━━━━━━━━━━━━━━ 21s 18ms/step - loss: 8.0212e-04
Epoch 6/20
1092/1092 ━━━━━━━━━━━━━━━━━━━━ 20s 18ms/step - loss: 6.6422e-04
Epoch 7/20
1092/1092 ━━━━━━━━━━━━━━━━━━━━ 20s 18ms/step - loss: 6.1612e-04
Epoch 8/20
1092/1092 ━━━━━━━━━━━━━━━━━━━━ 21s 19ms/step - loss: 5.4265e-04
Epoch 9/20
1092/1092 ━━━━━━━━━━━━━━━━━━━━ 23s 21ms/step - loss: 5.0552e-04
Epoch 10/20
1092/1092 ━━━━━━━━━━━━━━━━━━━━ 21s 19ms/step - loss: 4.6541e-04
Epoch 11/20
1092/1092 ━━━━━━━━━━━━━━━━━━━━ 20s 18ms/step - loss: 4.3341e-04
Epoch 12/20
1092/1092 ━━━━━━━━━━━━━━━━━━━━ 19s 17ms/step - loss: 4.1025e-04
Epoch 13/20
1092/1092 ━━━━━━━━━━━━━━━━━━━━ 19s 17ms/step - loss: 3.9737e-04
Epoch 14/20
1092/1092 ━━━━━━━━━━━

In [14]:
print(f"\n✅ Eğitim tamamlandı. En iyi Eğitim Kaybı (MSE): {min(history.history['loss']):.6f}")


✅ Eğitim tamamlandı. En iyi Eğitim Kaybı (MSE): 0.000374


In [15]:
# Tahminleri Üretme ve Sınırlandırma
predictions = model.predict(X_test_scaled).flatten()
predictions = np.clip(predictions, 0.0, 1.0) # Olasılık değerini 0 ile 1 arasına sabitleme


23291/23291 ━━━━━━━━━━━━━━━━━━━━ 39s 2ms/step


In [16]:
# Kaggle Formatsal Çıktı Dosyasını Hazırlama
submission = pd.DataFrame({
    'id': test['id'],
    'FloodProbability': predictions
})

In [17]:
submission.to_csv('submission.csv', index=False)